# 01. Single-Agent Foundations

A first runnable notebook for the OpenAI Agents SDK. It uses a tiny digital humanities example: extracting people, places, dates, and evidence from short historical texts.

## Run locally
1. `uv sync`
2. set `OPENAI_API_KEY`
3. run the cells top to bottom

## Google Colab
1. Open a new Colab notebook
2. Run `!pip install openai-agents pandas python-dotenv`
3. If you cloned the repo, the `data/` folder is already available. If you only copied the notebook, upload `data.zip` from the repository root and extract it in the next setup cell. You can also replace the contents of `data/` with your own `.txt` files.
4. Add your API key with `os.environ["OPENAI_API_KEY"] = "..."` or use Colab secrets

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Concepts
- Agent
- Tools
- Structured output
- Reproducibility

## Why this notebook first
The SDK README describes agents as LLMs configured with instructions, tools, guardrails, and handoffs. For beginners, the simplest starting point is one agent with one clear job. That keeps the mental model small: you tell the agent what to do, optionally give it tools, and inspect the result.

In DH work, that often means one task such as extracting names from a letter, normalizing a place name, or turning a short archival excerpt into structured metadata.

## Dataset
Any `.txt` files stored in `data/` are loaded directly in the notebook.

## Colab data setup

Use this cell only when running in Google Colab and the `data/` folder is not already present. Upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the cell so the notebook can load the sample documents.

## Further reading
- Agents: https://openai.github.io/openai-agents-python/agents
- Tools: https://openai.github.io/openai-agents-python/tools/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [64]:
!pip install openai-agents pandas python-dotenv

In [65]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, set_tracing_export_api_key, trace


In [66]:
import os
from google.colab import userdata

# Pega a chave salva nos Segredos do Colab
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [67]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)

if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)

    if zip_path is not None:
        import zipfile

        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)

if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


In [68]:
def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

DOCUMENTS = load_documents()

print('loaded', len(DOCUMENTS), 'documents')

loaded 3 documents


In [69]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: A small tool

Tools are ordinary Python functions that the agent can call when it needs to take an action or transform data. The SDK README groups tools with functions, MCP tools, and hosted tools; in this notebook we begin with the most familiar kind, a local Python function.

In practice, tools are useful when the model should not guess or improvise. A normalization function, a lookup function, or a validation function makes the workflow more reliable and more transparent.

In [70]:
def normalize_place_name(place: str) -> str:
    """Normalize a place name for historical records."""
    return place.strip().title()

print(normalize_place_name(' madrid '))

Madrid


## Step 1b: Turn the helper into a tool

Now we wrap the same function as an SDK tool.

In [71]:
normalize_place_name_tool = function_tool(normalize_place_name)
normalize_place_name_tool

FunctionTool(name='normalize_place_name', description='Normalize a place name for historical records.', params_json_schema={'properties': {'place': {'title': 'Place', 'type': 'string'}}, 'required': ['place'], 'title': 'normalize_place_name_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7c3b8f2e4110>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

## Step 2: Define the output shape

Using a structured output makes it easy to inspect results, compare runs, and grade student work. The SDK supports output types such as dataclasses, Pydantic models, and TypedDicts. For teaching, a small dataclass keeps the schema readable while still showing the discipline of structured data.

This also helps make the difference between free-form prose and machine-readable research data easier to see.

In [72]:
from dataclasses import dataclass, field

@dataclass
class Extraction:
    people: list[str] = field(default_factory=list)
    places: list[str] = field(default_factory=list)
    dates: list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)


## Step 3: Build the agent

The agent is intentionally narrow: it extracts evidence from short texts. The SDK README emphasizes that agents are LLMs configured with instructions, tools, guardrails, and handoffs. Here we only use instructions, one tool, and a structured output so the core pattern stays visible.

A good teaching habit is to write instructions in terms of constraints, not just goals. For example: be conservative, avoid guessing, and return evidence.

In [73]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Extract people, places, dates, and evidence from short historical text. '
        'Be conservative: if you are unsure, leave a field empty rather than guessing.'
    ),
    tools=[normalize_place_name_tool],
    output_type=Extraction,
)

archive_agent

Agent(name='Archive Extractor', handoff_description=None, tools=[FunctionTool(name='normalize_place_name', description='Normalize a place name for historical records.', params_json_schema={'properties': {'place': {'title': 'Place', 'type': 'string'}}, 'required': ['place'], 'title': 'normalize_place_name_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7c3b8f2e4110>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)], mcp_servers=[], mcp_config={}, instructions='Extract people, places, dates, and evidence from short historical text. Be conservative: if you are unsure, leave a field empty rather than guessing.', prompt=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=Non

## Step 4: Run the agent

Set `text` to one of the sample rows. In notebooks, use `await Runner.run(...)` instead of `run_sync(...)` because Jupyter already has an event loop. That is the most direct way to show the end-to-end flow: input in, structured output out.

If you want to discuss tracing later, this is also the point where you can explain that each run can be inspected and debugged.

In [74]:
text = DOCUMENTS[0]['text']
with trace('DH extraction demo'):
    result = await Runner.run(archive_agent, text)
print(result.final_output)


Extraction(people=['Maria Gomez'], places=['Madrid', 'Valencia'], dates=['4 April 1871', '1871'], evidence=['"Madrid, 4 April 1871"', '"In 1871, Maria Gomez wrote from Madrid to her brother in Valencia about the exhibition and the costs of travel."', '"she hoped the spring weather would make the journey easier when she next visited."'])


## Step 5: Put results in a table

This is useful for classroom discussion and for comparing outputs across prompt versions. In DH teaching, a table makes the artifact feel like research data rather than a black-box answer.

In [75]:
rows = [
    {'field': 'people', 'value': result.final_output.people},
    {'field': 'places', 'value': result.final_output.places},
    {'field': 'dates', 'value': result.final_output.dates},
    {'field': 'evidence', 'value': result.final_output.evidence},
]
pd.DataFrame(rows)

,field,value
0,people,[Maria Gomez]
1,places,"[Madrid, Valencia]"
2,dates,"[4 April 1871, 1871]"
3,evidence,"[""Madrid, 4 April 1871"", ""In 1871, Maria Gomez..."


## Exercise

Change the instructions so the agent extracts institutions instead of people. Use the printing house notice and compare results.

## Solution

Replace `people` with `institutions` in the output schema, update the instructions to say `extract institutions`, and rerun on another document from `DOCUMENTS`.